OpenAIEmbeddings

In [ ]:
# [목적] OpenAI 임베딩 예제를 실행할 환경과 LangSmith 추적을 준비합니다.
# .env 파일에서 API 키를 읽고, 실행 기록을 Chapter11-Embeddings 프로젝트로 전송합니다.
# API 키를 코드에 직접 쓰지 않고 안전하게 사용하며, 이후 임베딩 호출을 확인하기 위해 필요합니다.
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("Chapter11-Embeddings")

In [ ]:
# [목적] 문장을 숫자 벡터로 변환할 OpenAI 임베딩 모델을 생성합니다.
# OpenAIEmbeddings는 텍스트의 의미를 비교할 수 있는 숫자 목록을 만드는 LangChain 클래스입니다.
# 이후 질문·문서 임베딩과 문장 유사도 계산에서 이 객체를 사용합니다.
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
# [목적] 임베딩 변환 결과를 확인할 예제 문장을 준비합니다.
# text 변수에 한 문장을 저장한 뒤, 다음 셀에서 모델에 전달합니다.
# 같은 문장을 질문과 문서 방식으로 각각 변환해 결과 형태를 비교하기 위해 필요합니다.
text = "임베딩 테스트를 하기 위한 샘플 문장입니다."

In [ ]:
# [목적] 예제 문장을 검색어용 임베딩 벡터로 변환합니다.
# embed_query는 검색할 질문이나 사용자의 입력 한 건을 숫자 벡터로 바꿉니다.
# 만들어진 query_result는 문서 벡터와의 유사도를 계산할 때 사용할 수 있습니다.
query_result = embeddings.embed_query(text)

In [ ]:
# [목적] 검색어 임베딩 벡터의 숫자 개수를 확인합니다.
# len은 벡터를 이루는 숫자 차원 수를 반환합니다.
# 모델이 기본 설정에서 어느 크기의 벡터를 만드는지 확인하기 위해 사용합니다.
len(query_result)

In [ ]:
# [목적] 검색어 임베딩 벡터의 일부 값을 살펴봅니다.
# [:5]는 벡터 전체 중 앞의 다섯 숫자만 가져오는 슬라이싱 문법입니다.
# 벡터가 실제 숫자 목록으로 생성되었는지 간단히 확인하기 위해 사용합니다.
query_result[:5]

In [ ]:
# [목적] 여러 문서를 한 번에 문서용 임베딩 벡터로 변환합니다.
# embed_documents는 문장 목록을 받아 각 문장에 대응하는 벡터 목록을 반환합니다.
# 검색 시스템에서 비교 대상이 될 문서 데이터를 준비하는 방식입니다.
doc_result = embeddings.embed_documents(
    [text, text, text, text]
)

In [ ]:
# [목적] 첫 번째 문서 임베딩 벡터의 앞부분을 확인합니다.
# doc_result[0]은 문서 목록 중 첫 문장의 벡터이며, [:5]로 일부 숫자만 표시합니다.
# 질문용 임베딩과 문서용 임베딩이 같은 숫자 벡터 형태인지 확인하기 위해 사용합니다.
doc_result[0][:5]

In [ ]:
# [목적] 첫 번째 문서 임베딩 벡터의 차원 수를 확인합니다.
# 문서 벡터의 숫자 개수를 세어 검색어 벡터와 같은 길이인지 비교합니다.
# 서로 유사도를 계산하려면 비교하는 두 벡터의 길이가 같아야 합니다.
len(doc_result[0])

In [ ]:
# [목적] 차원 수를 지정한 임베딩 모델로 벡터 크기를 조절합니다.
# dimensions=1024는 기본 벡터보다 작은 1,024개 숫자로 결과를 만들도록 요청합니다.
# 저장 공간이나 검색 비용을 줄이면서 임베딩을 활용하는 방법을 확인합니다.
embeddings_1024 = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=1024)

len(embeddings_1024.embed_documents([text])[0])

In [ ]:
# [목적] 여러 문장을 임베딩하고 의미가 얼마나 가까운지 비교할 데이터를 준비합니다.
# cosine_similarity는 두 숫자 벡터의 방향이 비슷한 정도를 계산하는 함수입니다.
# 문장 목록을 1,024차원 벡터로 바꿔 다음 셀의 유사도 비교에 사용합니다.
from sklearn.metrics.pairwise import cosine_similarity

sentence1 = "안녕하세요? 반갑습니다."
sentence2 = "안녕하세요? 반갑습니다!"
sentence3 = "안녕하세요? 만나서 반가워요."
sentence4 = "Hi, nice to meet you."
sentence5 = "I like to eat apples."

sentences = [sentence1, sentence2, sentence3, sentence4, sentence5]

embedded_sentences = embeddings_1024.embed_documents(sentences)

In [ ]:
# [목적] 두 임베딩 벡터의 코사인 유사도를 한 값으로 계산하는 함수를 만듭니다.
# cosine_similarity는 목록 형태의 입력을 받으므로 각 벡터를 대괄호로 한 번 감쌉니다.
# 반환값 [0][0]으로 계산 결과 중 두 문장 사이의 유사도 숫자만 꺼냅니다.
def similarity (a, b):
    return cosine_similarity([a], [b])[0][0]

In [ ]:
# [목적] 모든 문장 쌍의 의미 유사도를 계산해 출력합니다.
# 두 반복문이 문장 벡터를 하나씩 비교하고, i < j 조건으로 같은 조합의 중복 계산을 막습니다.
# 출력된 점수를 통해 비슷한 문장이 더 높은 유사도를 갖는지 확인합니다.
for i, sentence in enumerate(embedded_sentences):
    for j, other_sentence in enumerate(embedded_sentences):
        if i < j:
            print(
                f"[유사도 {similarity(sentence, other_sentence):.4f}] {sentences[i]} \t <=====> \t {sentences[j]}"
            )